# Exploratory Data Analysis (EDA) — Bank Loan Approval Dataset

This notebook performs a comprehensive exploratory data analysis on the **Bank Loan Approval and Customer Risk Analysis** dataset (`architsharma01/loan-approval-prediction-dataset`).

### Key Analysis Objectives:
1. Understand dataset shape, feature types, and verify absence of missing values.
2. Investigate the target variable distribution (`loan_status`).
3. Analyze key predictors of loan approval (e.g. CIBIL score, loan term, income, assets).
4. Perform feature correlation and interaction analysis.
5. Formulate hypotheses for preprocessing, modeling, and risk segmentation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"font.size": 11, "figure.autolayout": True})

# Load the raw dataset
df = pd.read_csv('../data/raw/loan_approval_dataset.csv')
# Strip any leading/trailing whitespace from column names and string values
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 1. Data Integrity & Summary Statistics
Check for missing values, column data types, and five-number summary.

In [ ]:
print("Missing Values Check:")
print(df.isnull().sum())

print("\nData Types:")
print(df.dtypes)

df.describe().T

## 2. Target Variable Analysis: `loan_status`
Evaluate class balance between Approved and Rejected applications.

In [ ]:
status_counts = df['loan_status'].value_counts()
status_pct = df['loan_status'].value_counts(normalize=True) * 100

summary_target = pd.DataFrame({'Count': status_counts, 'Percentage (%)': status_pct.round(2)})
display(summary_target)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(status_counts.index, status_counts.values, color=['#2ecc71', '#e74c3c'], width=0.5, edgecolor='black')
for bar in bars:
    h = bar.get_height()
    pct = (h / len(df)) * 100
    ax.annotate(f"{h}\n({pct:.1f}%)", (bar.get_x() + bar.get_width() / 2, h / 2),
                ha='center', va='center', color='white', fontweight='bold')
ax.set_title("Target Distribution (Loan Approval)", fontsize=13, fontweight='bold')
ax.set_ylabel("Count")
plt.show()

## 3. Credit Score (CIBIL) Analysis
Analyze how CIBIL credit score differentiates Approved vs. Rejected applicants.

In [ ]:
display(df.groupby('loan_status')['cibil_score'].describe().round(2))

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.histplot(
    data=df,
    x='cibil_score',
    hue='loan_status',
    kde=True,
    palette={'Approved': '#2ecc71', 'Rejected': '#e74c3c'},
    bins=30,
    alpha=0.6,
    edgecolor='black',
    ax=ax
)
ax.axvline(550, color='gray', linestyle='--', linewidth=1.5, label='Risk Threshold (550)')
ax.set_title("CIBIL Credit Score Distribution by Loan Status", fontweight='bold')
ax.legend(title='Loan Status')
plt.show()

## 4. Financial Features & Derived Ratios
Examine income, loan amount, and asset holdings.

In [ ]:
df['total_assets'] = (
    df['residential_assets_value']
    + df['commercial_assets_value']
    + df['luxury_assets_value']
    + df['bank_asset_value']
)
df['loan_to_income'] = df['loan_amount'] / df['income_annum']
df['asset_to_loan'] = df['total_assets'] / df['loan_amount']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
sns.boxplot(x='loan_status', y=df['income_annum'] / 1e5, data=df, ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title("Annual Income (Lakhs INR)")
axes[0].set_xlabel("Loan Status")

sns.boxplot(x='loan_status', y=df['loan_amount'] / 1e5, data=df, ax=axes[1], palette=['#2ecc71', '#e74c3c'])
axes[1].set_title("Loan Amount (Lakhs INR)")
axes[1].set_xlabel("Loan Status")

sns.boxplot(x='loan_status', y=df['total_assets'] / 1e5, data=df, ax=axes[2], palette=['#2ecc71', '#e74c3c'])
axes[2].set_title("Total Assets (Lakhs INR)")
axes[2].set_xlabel("Loan Status")
plt.show()

## 5. Loan Term, Education & Employment Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Approval rate by loan term
term_app = df.groupby('loan_term')['loan_status'].apply(lambda x: (x == 'Approved').mean())
sns.barplot(x=term_app.index, y=term_app.values, color='#3498db', ax=axes[0], edgecolor='black')
axes[0].set_title("Approval Rate by Loan Term (Years)", fontweight='bold')
axes[0].set_ylabel("Approval Rate")
axes[0].set_ylim(0, 1.0)

# Approval rate by education and employment
edu_emp = df.groupby(['education', 'self_employed'])['loan_status'].apply(lambda x: (x == 'Approved').mean()).unstack()
edu_emp.plot(kind='bar', ax=axes[1], colormap='viridis', edgecolor='black')
axes[1].set_title("Approval Rate by Education & Self-Employed Status", fontweight='bold')
axes[1].set_ylabel("Approval Rate")
axes[1].set_ylim(0, 1.0)
axes[1].legend(title='Self Employed')
axes[1].tick_params(axis='x', rotation=0)

plt.show()

## 6. Correlation Matrix

In [ ]:
df_corr = df.copy()
df_corr['loan_status_num'] = (df_corr['loan_status'] == 'Approved').astype(int)
df_corr['education_num'] = (df_corr['education'] == 'Graduate').astype(int)
df_corr['self_employed_num'] = (df_corr['self_employed'] == 'Yes').astype(int)

numeric_cols = [
    'cibil_score', 'loan_status_num', 'loan_term', 'income_annum',
    'loan_amount', 'total_assets', 'loan_to_income', 'asset_to_loan',
    'education_num', 'self_employed_num', 'no_of_dependents'
]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df_corr[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title("Correlation Matrix", fontweight='bold')
plt.show()

## 7. Key Findings & Modeling Strategy

1. **Cleanliness**: 0 missing values across all 4,269 rows. Column names and string values contain leading whitespace which will be cleaned in preprocessing.
2. **Class Balance**: 62.2% Approved (2,656) vs 37.8% Rejected (1,613). Moderate imbalance — standard stratified sampling is appropriate.
3. **Primary Predictor**: `cibil_score` is the dominant feature (correlation 0.77). Applicants with CIBIL >= 600 have an overwhelming approval rate, while CIBIL < 550 faces near-universal rejection.
4. **Secondary Predictors**: Longer `loan_term` moderately increases rejection probability. Income and asset values provide critical collateral context for risk scoring.
5. **Feature Engineering**: Creating `total_assets`, `loan_to_income` ratio, and `asset_to_loan` coverage provides strong financial interpretability for both approval classification and risk tiering.
